# sorted-computational-graph — ex3: sort a diamond DAG with one shared root

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sorted-computational-graph`. Running the final beacon cell reports progress against the `Backprop: Sorted computation graph` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Sorted computation graph` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sorted-computational-graph`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sorted-computational-graph"
DD_SUBTOPIC = "Backprop: Sorted computation graph"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Diamond compute graph — one shared leaf, two paths, single visit

Ex1 sorted a simple linear chain; ex2 handled a multi-depth shared-leaf graph. The deepening move pins down the canonical 'diamond' DAG pattern, which appears anywhere a single tensor flows through two parallel ops and merges back:

```
        a       (leaf)
       / \
      b   c     (b = log(a), c = neg(a))
       \ /
        d       (d = b * c)
```

**The four invariants the sort must satisfy.**
1. Each of `{a, b, c, d}` appears EXACTLY once.
2. `d` is first in the reverse-topo order.
3. `b` and `c` BOTH appear before `a` (parent-before-child holds across BOTH paths through the diamond, not just one).
4. An unrelated leaf `z` (not consumed by `d`) is NOT visited.

**Why the `perm` set is the load-bearing part.** Without `id(...)` membership de-duplication, the second visit to `a` (via `c`'s parent list) would add it again. The sort would then have either `a` duplicated OR `b`/`c` placed wrongly. The diamond is the smallest graph that exposes this bug — a simple chain doesn't.

**Generalisation.** Any DAG with shared ancestry is a 'union of diamonds'. Getting the diamond right is sufficient to get arbitrary DAGs right.

### Exercise 3 — sort a diamond DAG with one shared root

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply topological sort over a diamond compute graph a -> {b, c} -> d to produce a single-visit reverse-topo order where d is first, b and c both precede a, and unrelated nodes are not visited.
> Keywords: diamond, dag, topo-sort, shared-root, deduplication
> ```

**KCs targeted:** `diamond-graph-ordering`, `perm-set-prevents-duplicates`

Implement `topological_sort(node, get_children)` and `sorted_computational_graph(tensor)`.

Spec:
- `topological_sort(node, get_children)` does a DFS with a `perm` set keyed by `id(child)` to avoid revisits. Returns the post-order (leaves first, end-node last).
- `sorted_computational_graph(tensor)` calls `topological_sort` with `get_children = lambda n: list(n.recipe.parents.values()) if n.recipe else []`, then REVERSES the result so the end-node is first.

The test graph is a diamond:

```
    a         (leaf, single root)
   / \
  b   c      (b = log(a), c = neg(a))
   \ /
    d        (d = b * c, end-node)
```

Verify:
1. Result length is exactly 4 (one entry per node: a, b, c, d).
2. `d` is at index 0 (reverse-topo: end-node first).
3. `a` is at index 3 (leaf comes last in reverse-topo).
4. `b` and `c` both come before `a` and after `d`.
5. An unrelated leaf `z` (constructed but not connected to the graph) does NOT appear in the result.
6. The graph also handles a multi-output extension: appending a second end-node `e = b + c` (sharing b and c with d) and walking from `e` gives the right order with `e` first.

In [ ]:
def topological_sort(node, get_children):
    result = []
    perm = set()
    def visit(cur):
        if id(cur) in perm:
            return
        perm.add(id(cur))
        for child in get_children(cur):
            visit(child)
        result.append(cur)
    visit(node)
    return result


def sorted_computational_graph(tensor):
    def get_children(n):
        if n.recipe is None:
            return []
        return list(n.recipe.parents.values())
    return topological_sort(tensor, get_children)[::-1]


<details><summary>Solution</summary>

```python
def topological_sort(node, get_children):
    result = []
    perm = set()
    def visit(cur):
        if id(cur) in perm:
            return
        perm.add(id(cur))
        for child in get_children(cur):
            visit(child)
        result.append(cur)
    visit(node)
    return result


def sorted_computational_graph(tensor):
    def get_children(n):
        if n.recipe is None:
            return []
        return list(n.recipe.parents.values())
    return topological_sort(tensor, get_children)[::-1]
```

**The `perm` set keyed by `id` is load-bearing.** Without deduplication, the diamond's leaf `a` would be visited twice (once via `b`'s parents, once via `c`'s parents). The post-order list would then have `a` appearing twice — breaking the exact-once invariant.

**Why `id(...)` and not the tensor itself.** MiniTensors aren't hashable by value (they hold mutable `.array`). Using `id` is the standard idiom — it's stable for the object's lifetime and gives O(1) membership.

**Post-order + reverse is the canonical reverse-topo trick.** DFS post-order has leaves first; reversing puts the end-node first. The reverse pass consumes this list head-to-tail, guaranteeing that by the time it processes a node, all of that node's CHILDREN (downstream consumers) have already been visited and their grads computed. That's the invariant that makes single-pass reverse-mode correct.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()